# Deteksi Plat Nomor: Jogja vs Non-Jogja dengan YOLOv11

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hisyamardnsyahx/deteksi-plat-nomor-jogja/blob/main/colab_yolo11_plat_jogja.ipynb)

Notebook ini mengklasifikasikan plat nomor kendaraan Indonesia
menjadi Jogja atau non-Jogja. Patokannya kode wilayah AB, yang
mencakup Kota Yogyakarta, Sleman, Bantul, Kulon Progo, dan
Gunungkidul. Kode wilayah lain dihitung sebagai non-Jogja.

Soalnya bukan sekadar menemukan kotak plat. Bentuk plat AB dan
non-AB sama, jadi yang membedakan hanyalah huruf di awal plat.
Model di sini belajar pola huruf itu dari piksel, dan karena itu
ukuran gambar dibuat besar dan flip dimatikan. Dua keputusan itu
mengikuti langsung dari soalnya, bukan sekadar tuning.

Hasil yang diharapkan, dari run referensi 4 November 2026 di Colab
dengan GPU Tesla T4, 100 epoch selesai dalam sekitar 24 menit,
tanpa early stopping dan tanpa kehabisan memori:

| Split | Gambar | Precision | Recall | mAP50 | mAP50-95 |
|---|---:|---:|---:|---:|---:|
| Validasi | 62 | 0,985 | 0,978 | 0,995 | 0,787 |
| Test | 61 | 0,997 | 1,000 | 0,995 | 0,800 |

Pada test, dengan IoU minimal 0,5 dan confidence 0,25, hasilnya
33 Jogja benar, 28 non-Jogja benar, tidak ada yang salah kelas,
tidak ada yang terlewat, dan ada 5 false positive berupa kotak
tambahan.

Isi notebook ini: (1) lingkungan dan instalasi, (2) data, mulai
dari unduh sampai relabel, (3) baseline dengan model pretrained
COCO, (4) training, (5) evaluasi, (6) artefak, dan (7) lampiran.

Cara menjalankannya: buka di Google Colab, atur Runtime, Change
runtime type, Hardware accelerator ke T4 GPU, lalu Runtime, Run
all. Jalankan sel berurutan dan jangan mengubah urutan unduh,
audit, relabel, lalu training. Bila ada konfigurasi yang diubah,
misalnya batch karena kehabisan memori, catat perubahannya untuk
laporan.

Sumber datanya adalah Roboflow Universe, Universitas Teknologi
Yogyakarta, "Deteksi Plat Nomor Kendaraan 2", dataset versi 1,
16 Desember 2023, lisensi CC BY 4.0. Halaman sumber tidak
menyediakan paper, metrik acuan, atau bobot model, sehingga
baseline di notebook ini dibangun sendiri.


## 1. Lingkungan dan instalasi

Sel pertama memastikan runtime benar-benar memakai GPU. Training
100 epoch pada ukuran gambar 960 tidak realistis di CPU. Sebagai
gambaran, pilot lokal tanpa GPU butuh sekitar 3 menit per epoch
pada ukuran 640.


In [ ]:
# 1. Cek GPU — HARUS ada GPU, kalau tidak, ganti runtime ke T4 dulu
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU tidak terdeteksi. Runtime > Change runtime type > T4 GPU, lalu jalankan ulang.')


In [ ]:
# 2. Instalasi
!pip install -q ultralytics gdown
from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__)


## 2. Data: unduh, audit, dan relabel

Dataset diunduh dari Google Drive penulis dalam bentuk ZIP YOLOv11
sekitar 53 MB, lalu diaudit dan direlabel. Urutan ini tidak boleh
dibalik, karena training memakai file data dua kelas yang baru
terbentuk pada sel relabel.


In [ ]:
# 3. Unduh dataset dari Drive (ZIP YOLOv11, 53 MB) dan ekstrak
import gdown, zipfile, pathlib
FILE_ID = '1fWxM6y6dqQCj-3FkCNSuKx7uqLe9j64Y'  # Deteksi Plat Nomor Kendaraan 2.v1i.yolov11.zip
zip_path = '/content/dataset.zip'
gdown.download(id=FILE_ID, output=zip_path, quiet=False)
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/roboflow')
print('data.yaml asli:')
print(pathlib.Path('/content/roboflow/data.yaml').read_text())
import glob
for split in ('train','valid','test'):
    print(split, len(glob.glob(f'/content/roboflow/{split}/images/*')))


In [ ]:
# 4. Audit Jogja/non-Jogja dari nama file (+ 1 koreksi manual terverifikasi visual)
# Nama file mengkodekan teks plat, mis. K1_AB2430VL = AB 2430 VL.
# Pengecualian: K4_1938WA tidak membawa prefix di namanya, padahal fotonya AB 1938 WA.
import csv, re, pathlib
from collections import Counter
MANUAL = {'K4_1938WA_jpg.rf.e572eadcb5e7b50790ef081d9f4370d1.jpg': ('AB', 'plat-jogja')}
root = pathlib.Path('/content/roboflow')
rows = []
for split in ('train','valid','test'):
    for img in sorted((root/split/'images').glob('*')):
        if img.suffix.lower() not in {'.jpg','.jpeg','.png'}:
            continue
        if img.name in MANUAL:
            prefix, label = MANUAL[img.name]
            rows.append({'image': img.name, 'split': split, 'prefix': prefix, 'label': label, 'source': 'manual_visual', 'notes': 'nama file tanpa prefix; dibaca dari foto'})
        else:
            m = re.match(r'^[A-Z0-9]+_([A-Z]+)\d', img.name)
            assert m, f'Token tidak ter-parse: {img.name} — periksa manual, jangan ditebak'
            prefix = m.group(1)
            rows.append({'image': img.name, 'split': split, 'prefix': prefix,
                         'label': 'plat-jogja' if prefix == 'AB' else 'plat-non-jogja',
                         'source': 'filename', 'notes': ''})
with open('/content/audit_jogja.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=['image','split','prefix','label','source','notes']); w.writeheader(); w.writerows(rows)
print('Total:', len(rows))
print(Counter((r['split'], r['label']) for r in rows))
assert len(rows) == 614, 'Total harus 614 sesuai dataset sumber'


In [ ]:
# 5. Relabel 1 kelas -> 2 kelas, split asli dipertahankan
# Alasan split dipertahankan: audit lokal menunjukkan nol token plat yang bocor antar split
# dan proporsi Jogja sudah seimbang (train 266/225, valid 30/32, test 33/28).
import csv, shutil, pathlib
MAP = {'plat-jogja': 0, 'plat-non-jogja': 1}
src = pathlib.Path('/content/roboflow'); dst = pathlib.Path('/content/jogja_2cls')
for r in csv.DictReader(open('/content/audit_jogja.csv')):
    assert (r['label'] == 'plat-jogja') == (r['prefix'] == 'AB'), f'Label/prefix tidak konsisten: {r}'
    img = src/r['split']/'images'/r['image']
    lbl_src = src/r['split']/'labels'/(pathlib.Path(r['image']).stem + '.txt')
    out_img = dst/r['split']/'images'/r['image']; out_lbl = dst/r['split']/'labels'/(pathlib.Path(r['image']).stem + '.txt')
    out_img.parent.mkdir(parents=True, exist_ok=True); out_lbl.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(img, out_img)
    lines = []
    for ln in lbl_src.read_text().splitlines():
        if not ln.strip(): continue
        p = ln.split(); w_, h_ = float(p[3]), float(p[4])
        if w_ < 0.01 or h_ < 0.01:
            print('Kotak degeneratif dibuang:', r['image']); continue
        lines.append(f"{MAP[r['label']]} " + ' '.join(p[1:]))
    out_lbl.write_text('\n'.join(lines) + '\n')
(dst/'data.yaml').write_text('path: /content/jogja_2cls\ntrain: train/images\nval: valid/images\ntest: test/images\nnc: 2\nnames:\n  0: plat-jogja\n  1: plat-non-jogja\n')
print('data.yaml 2 kelas:')
print((dst/'data.yaml').read_text())


## 3. Baseline: YOLOv11 pretrained COCO pada test

Ekspektasinya, model COCO mendeteksi kendaraan, bukan plat, karena
COCO tidak memiliki kelas plat nomor. Sel ini menunjukkan kenapa
fine-tuning wajib dilakukan sebelum ada klaim deteksi plat. Pada
run referensi, hasilnya 284 deteksi pada 61 gambar test: motor 175,
mobil 90, orang 16, truk 1, pot tanaman 1, ransel 1, dan nol plat.


In [ ]:
# 6. Baseline A3 (opsional, cepat di GPU): pretrained COCO di data test
# Ekspektasi: model COCO mendeteksi kendaraan (mobil/motor), BUKAN plat — COCO tidak punya kelas plat.
# Sel ini membuktikan kenapa fine-tune wajib, sebelum klaim deteksi plat.
import csv
from ultralytics import YOLO
model0 = YOLO('yolo11n.pt')
rows = []
for img in sorted(pathlib.Path('/content/roboflow/test/images').glob('*')):
    if img.suffix.lower() not in {'.jpg','.jpeg','.png'}: continue
    res = model0.predict(str(img), imgsz=640, conf=0.25, verbose=False)[0]
    dets = [(model0.names[int(c)], round(float(s),3)) for c,s in zip(res.boxes.cls, res.boxes.conf)] if res.boxes is not None else []
    rows.append({'image': img.name, 'n_det': len(dets), 'detections': ';'.join(f'{n}:{s}' for n,s in dets)})
with open('/content/baseline_pretrained.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=['image','n_det','detections']); w.writeheader(); w.writerows(rows)
from collections import Counter
c = Counter()
for r in rows:
    for d in r['detections'].split(';'):
        if d: c[d.rsplit(':',1)[0]] += 1
print('Test:', len(rows), 'gambar, total deteksi:', sum(int(r['n_det']) for r in rows), 'per kelas:', dict(c))


## 4. Training penuh dua kelas (sel utama)

Konfigurasi di bawah ini adalah konfigurasi yang dilaporkan
sebagai hasil utama. Ubah hanya bila ada alasan yang dicatat
untuk laporan.

| Parameter | Nilai | Alasan |
|---|---|---|
| Model awal | `yolo11n.pt` (pretrained COCO) | Transfer learning; naik ke `yolo11s.pt` hanya bila mAP validasi mentok |
| Data | `/content/jogja_2cls/data.yaml` | Dataset relabel dua kelas, split asli dipertahankan |
| Epoch / patience / seed | 100 / 50 / 42 | Run referensi selesai 100 dari 100 epoch tanpa early stopping |
| Ukuran gambar / batch / workers | 960 / 16 / 2 | 960 agar huruf prefix terbaca; batch 16 muat di T4 pada run referensi |
| Augmentasi | `mosaic=1.0`, `close_mosaic=10` | Mosaic dimatikan 10 epoch terakhir |
| Flip | `fliplr=0.0`, `flipud=0.0` | Flip horizontal mencerminkan huruf plat, sehingga AB yang dipelajari bukan lagi AB yang ada di jalan |

Bila GPU kehabisan memori (CUDA OOM), ubah `batch=16` menjadi
`batch=8` dan jalankan ulang sel training saja. Hasilnya tidak
lagi identik dengan run referensi, dan perbedaan itu perlu
dicatat.


In [ ]:
# 7. TRAINING PENUH 2 kelas Jogja vs non-Jogja
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data='/content/jogja_2cls/data.yaml', epochs=100, imgsz=960, batch=16,
            patience=50, seed=42, workers=2, fliplr=0.0, flipud=0.0,
            mosaic=1.0, close_mosaic=10,
            project='/content/runs', name='yolo11_jogja_colab')
print('Training selesai. Bobot terbaik ada di /content/runs/yolo11_jogja_colab/weights/best.pt')


## 5. Evaluasi: mAP validasi dan test, lalu confusion matrix

Dua evaluasi ini saling melengkapi. Nilai mAP mengukur kualitas
deteksi pada berbagai threshold, sedangkan confusion pada IoU
0,5 dan confidence 0,25 mengukur klasifikasi Jogja versus
non-Jogja per gambar.


In [ ]:
# 8. Evaluasi standar: valid (headline mAP) + test
from ultralytics import YOLO
best = YOLO('/content/runs/yolo11_jogja_colab/weights/best.pt')
print('=== VALID (headline mAP, conf validasi standar) ===')
print(best.val(data='/content/jogja_2cls/data.yaml', split='val', imgsz=960))
print('=== TEST ===')
print(best.val(data='/content/jogja_2cls/data.yaml', split='test', imgsz=960))


Cara membaca hasil evaluasi di atas: `best.val()` memakai
konfigurasi validasi standar Ultralytics untuk headline mAP, dan
konfigurasi itu jangan diganti dengan threshold confidence
deployment. Per kelas pada validasi run referensi, kelas Jogja
(30 instance) mencapai precision 0,97, recall 1,000, dan mAP50
0,995, sedangkan kelas non-Jogja (32 instance) mencapai precision
1,000, recall 0,956, dan mAP50 0,995. Nilai mAP50-95 yang lebih
rendah, sekitar 0,79, dibanding mAP50 yang sekitar 0,995 adalah
pola yang wajar: IoU yang lebih ketat menuntut kotak yang jauh
lebih presisi.


In [ ]:
# 9. Confusion matrix Jogja vs non-Jogja di test (IoU >= 0.5)
# Kotak GT tanpa pasangan = terlewat; prediksi tanpa pasangan = false positive;
# pasangan beda kelas = salah kelas. Ini metrik klasifikasi, pelengkap mAP.
import cv2
from ultralytics import YOLO
def iou(a,b):
    ix1, iy1 = max(a[0],b[0]), max(a[1],b[1]); ix2, iy2 = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0, ix2-ix1)*max(0, iy2-iy1)
    return inter/((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)
best = YOLO('/content/runs/yolo11_jogja_colab/weights/best.pt')
names = {0:'plat-jogja', 1:'plat-non-jogja'}
rows=[]; tp={'jogja':0,'non':0}; wrong=missed=fp=0
for img_path in sorted(pathlib.Path('/content/jogja_2cls/test/images').glob('*')):
    if img_path.suffix.lower() not in {'.jpg','.jpeg','.png'}: continue
    im = cv2.imread(str(img_path)); h,w = im.shape[:2]
    gt=[]
    lbl = pathlib.Path('/content/jogja_2cls/test/labels')/(img_path.stem+'.txt')
    for ln in lbl.read_text().splitlines():
        if ln.strip():
            p=ln.split(); cx,cy,bw,bh=[float(x) for x in p[1:5]]
            gt.append((int(p[0]), ((cx-bw/2)*w,(cy-bh/2)*h,(cx+bw/2)*w,(cy+bh/2)*h)))
    res = best.predict(str(img_path), imgsz=960, conf=0.25, verbose=False)[0]
    preds=[]
    if res.boxes is not None:
        for box,cls,conf in zip(res.boxes.xyxy, res.boxes.cls, res.boxes.conf):
            preds.append((int(cls), tuple(float(x) for x in box), float(conf)))
    preds.sort(key=lambda x:-x[2]); used=set()
    for gcls,gbox in gt:
        best_i, best_iou = -1, 0.5
        for pi,(pcls,pbox,_) in enumerate(preds):
            if pi in used: continue
            v=iou(gbox,pbox)
            if v>best_iou: best_i, best_iou = pi, v
        if best_i>=0:
            used.add(best_i); pcls=preds[best_i][0]
            if pcls==gcls:
                tp['jogja' if gcls==0 else 'non']+=1; rows.append({'image':img_path.name,'gt':names[gcls],'pred':names[pcls],'status':'benar'})
            else:
                wrong+=1; rows.append({'image':img_path.name,'gt':names[gcls],'pred':names[pcls],'status':'salah_kelas'})
        else:
            missed+=1; rows.append({'image':img_path.name,'gt':names[gcls],'pred':'','status':'terlewat'})
    for pi,(pcls,_,_) in enumerate(preds):
        if pi not in used:
            fp+=1; rows.append({'image':img_path.name,'gt':'','pred':names[pcls],'status':'false_positive'})
with open('/content/eval_test.csv','w',newline='') as f:
    w=csv.DictWriter(f, fieldnames=['image','gt','pred','status']); w.writeheader(); w.writerows(rows)
print(f"Benar Jogja: {tp['jogja']}, benar non-Jogja: {tp['non']}, salah kelas: {wrong}, terlewat: {missed}, false positive: {fp}")


## 6. Prediksi visual dan unduhan artefak

Prediksi pada seluruh gambar test disimpan sebagai gambar
beranotasi, dan artefak training diunduh sebagai ZIP. Sel ini
membutuhkan interaksi unduhan browser Colab, jadi izinkan unduhan
bila Colab memintanya.


In [ ]:
# 10. Contoh prediksi visual + unduh hasil
from ultralytics import YOLO
best = YOLO('/content/runs/yolo11_jogja_colab/weights/best.pt')
best.predict(source='/content/jogja_2cls/test/images', imgsz=960, conf=0.25,
             save=True, project='/content/runs', name='predict_test')
from google.colab import files
import shutil
shutil.make_archive('/content/hasil_jogja', 'zip', '/content/runs/yolo11_jogja_colab')
files.download('/content/hasil_jogja.zip')   # berisi best.pt + results.csv + grafik training
files.download('/content/eval_test.csv')
files.download('/content/audit_jogja.csv')
print('Unduhan dimulai. Simpan juga best.pt ke Drive kalau mau dipakai lagi (opsional):')
print('# from google.colab import drive; drive.mount("/content/drive")  # lalu copy best.pt ke Drive')


## 7. Artefak, keterbatasan, dan yang dikirim untuk laporan

Artefak yang diunduh dari sel di atas adalah `hasil_jogja.zip`,
yang berisi `best.pt`, `results.csv`, dan grafik training,
ditambah `eval_test.csv` yang berisi confusion per gambar dan
`audit_jogja.csv` yang berisi label hasil audit. Simpan ketiganya
bersama konfigurasi training, karena tanpa ketiganya angka di
laporan tidak bisa diverifikasi ulang.

Beberapa keterbatasan harus ikut ke laporan. Label dua kelas
diturunkan dari nama file Roboflow, bukan dari pembacaan manual
atas 614 foto. Satu pengecualian dikoreksi manual, yaitu file test
`K4_1938WA...jpg` yang namanya tanpa prefix, padahal fotonya
AB 1938 WA. Pemeriksaan konsistensi label terhadap nama file
menghasilkan nol ketidakcocokan selain file itu, tetapi
pemeriksaan tersebut bersifat sirkular dan hanya membuktikan
konsistensi, bukan kebenaran foto. Spot-check visual penuh sebesar
10 persen, yaitu 61 gambar, belum selesai; baru 7 foto yang dibaca
langsung dan semuanya cocok. Caveat label karena itu menyusut,
bukan tertutup. Angka-angka di sini juga hanya berlaku untuk
dataset Roboflow ini, yang platnya relatif besar di dalam frame.
Apakah modelnya bertahan pada foto HP di jalan belum teruji;
spesifikasinya ada di `docs/KEBUTUHAN_DATA.md` di repo.

Jalur OCR dua tahap, yaitu deteksi plat lalu crop lalu OCR prefix,
belum dievaluasi empiris di proyek ini. Yang dievaluasi hanya
YOLO dua kelas langsung, yang tidak mengeluarkan string teks plat.

Referensi pembanding terdekat, dengan detail di
`papers/DAFTAR_PUSTAKA.md` di repo, adalah Martadinata dkk. (2025,
Jurnal Sisfokom, YOLOv11 + EasyOCR), Kurniawan dan Sari (2025,
JAIC, YOLOv11 deteksi plat + karakter), serta Wijaya dan Arymurthy
(2025, JIKI, Gaussian YOLOv7). Tidak ada satu pun yang
mengklasifikasikan kode wilayah AB versus non-AB, dan itulah
celah kebaruan proyek ini.


## Lampiran A. Baseline satu kelas (opsional, di luar Run All)

Bagian ini bukan bagian dari Run All di atas. Jalankan hanya bila
diminta angka detektor satu kelas murni, di notebook salinan
terpisah atau setelah sel evaluasi, karena ini adalah training
kedua:

```python
from ultralytics import YOLO
m1 = YOLO('yolo11n.pt')
m1.train(data='/content/roboflow/data.yaml', epochs=100, imgsz=640, batch=16,
         patience=50, seed=42, workers=2, project='/content/runs', name='yolo11_plat1cls')
# Catatan: data.yaml asli Roboflow memakai path relatif; di Colab ganti dulu path-nya
# menjadi absolut /content/roboflow sebelum menjalankan sel ini.
```

## Lampiran B. Checklist pengiriman untuk laporan

1. `results.csv` dari folder training, berisi mAP50 dan mAP50-95
   validasi.
2. Angka test dari sel evaluasi, ditambah ringkasan confusion:
   benar, salah kelas, terlewat, dan false positive.
3. `best.pt` bila inference lokal diperlukan.
4. Catatan atas setiap perubahan konfigurasi dari yang dilaporkan,
   misalnya batch 16 menjadi 8 karena kehabisan memori, atau epoch
   yang berhenti lebih awal karena early stopping.
